In [1]:
!pip uninstall -y massi
!pip cache purge

Found existing installation: massi 0.1.0
Uninstalling massi-0.1.0:
  Successfully uninstalled massi-0.1.0
Files removed: 6 (1.2 MB)


In [2]:
!cd .. && pip install -v . --group=dev --group=observability --group=sandbox

Using pip 25.3 from /opt/conda/lib/python3.13/site-packages/pip (python 3.13)
Processing /home/jovyan/work
  Running command installing build dependencies
  Using pip 25.3 from /opt/conda/lib/python3.13/site-packages/pip (python 3.13)
    Obtaining dependency information for setuptools>=40.8.0 from https://files.pythonhosted.org/packages/9d/76/f789f7a86709c6b087c5a2f52f911838cad707cc613162401badc665acfe/setuptools-82.0.1-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.0 MB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.4 MB/s  0:00:00
  Installing build dependencies ... done
  Running command Getting requirements to build wheel
  running egg_info
  writing massi.egg-info/PKG-INFO
  writing dependency_links to massi.egg-info/dependency_links.txt
  writing requirements to massi.egg-info/requires.txt
  writing top-level names to massi.egg-info/top_level.txt
  reading manifest file 'massi.egg-info/SOURCES.txt'
  writing manifest file

In [3]:
# Setup observability
USE_OBSERVABILITY = True

if USE_OBSERVABILITY:
    import os
    from lmnr import Laminar
    
    Laminar.initialize(
        project_api_key=os.environ["LMNR_PROJECT_API_KEY"],
        base_url=f"http://{os.environ["HOST_MACHINE_URL_CORE"]}",
        http_port=int(os.environ["LMNR_PROJECT_HTTP_PORT"]),
        grpc_port=int(os.environ["LMNR_PROJECT_GRPC_PORT"]),
    )

1. **We will use the relay model for the sake of debugging convenience**

In [4]:
from massi.agents.massi import MassiScenarioFactory, Config, Dependencies, StandardIOPromptClient

In [5]:
from massi.runtimes.jupyter.ui import JupyterTextWidgetIOPromptClient

In [6]:
from massi.agents.massi import create_ollama_llm

base_config = Config(**{"model_url": "http://host.docker.internal:11434", "model_name": "lfm2.5-thinking"})
base_deps = Dependencies(llm_client=create_ollama_llm(base_config), prompt_client=JupyterTextWidgetIOPromptClient())

In [7]:
USE_RELAY_AI = False

if USE_RELAY_AI:
    from massi.agents.stunts import RelayChatModel
    relay_model = RelayChatModel()
    relay_deps = Dependencies(llm_client=RelayChatModel(), prompt_client=JupyterTextWidgetIOPromptClient())

In [8]:
USE_OPEN_AI = False

if USE_OPEN_AI:
    from massi.agents.massi import create_openai_llm
    openai_config = Config(**{"api_key": os.environ["OPENAI_API_TOKEN"], "model_name": "gpt-5-nano"})
    openai_deps = Dependencies(llm_client=create_openai_llm(openai_config), prompt_client=JupyterTextWidgetIOPromptClient())

In [9]:
if USE_RELAY_AI:
    print("... using mocking dependencies ...")
    factory = MassiScenarioFactory(base_config)
    graph = factory.resource_property_graph(relay_deps)
elif USE_OPEN_AI:
    print("... using OpenAI dependencies ...")
    factory = MassiScenarioFactory(openai_config)
    graph = factory.resource_property_graph(openai_deps)
else:
    print("... using standard dependencies ...")
    factory = MassiScenarioFactory(base_config)
    graph = factory.resource_property_graph(base_deps)

... using standard dependencies ...


In [10]:
import grandalf

graph.get_graph().print_ascii()

          +-----------+            
          | __start__ |            
          +-----------+            
                *                  
                *                  
                *                  
        +----------------+         
        | scan_directory |         
        +----------------+         
                .                  
                .                  
                .                  
       +-----------------+         
       | score_relevancy |         
       +-----------------+         
                *                  
                *                  
                *                  
       +-----------------+         
       | group_resources |         
       +-----------------+         
                *                  
                *                  
                *                  
+-------------------------------+  
| resources_to_description_file |  
+-------------------------------+  
                *           

In [11]:
import random

thread = {"configurable": {"thread_id": str(random.randint(1, 100))}}

In [12]:
import pprint

In [13]:
from massi.runtimes.jupyter.flow import stream_until_iterruption

In [14]:
from unittest.mock import patch

MOCK_RUN = False

In [15]:
stream_until_iterruption(graph, {}, thread)


==== new event ====
---- messages ----
[]
---- files ----
[   ResourceFile(file_location='data/.DS_Store', filename='.DS_Store', extension='', author='jovyan'),
    ResourceFile(file_location='data/README.txt', filename='README.txt', extension='.txt', author='jovyan'),
    ResourceFile(file_location='data/layer_data/PFAS sampling and purchasing and intake detail.csv', filename='PFAS sampling and purchasing and intake detail.csv', extension='.csv', author='jovyan'),
    ResourceFile(file_location='data/layer_data/WATER_SYSTEMS_IN_KENTUCKY.geojson', filename='WATER_SYSTEMS_IN_KENTUCKY.geojson', extension='.geojson', author='jovyan'),
    ResourceFile(file_location='data/layer_data/hot-spot_map.csv', filename='hot-spot_map.csv', extension='.csv', author='jovyan'),
    ResourceFile(file_location='data/layer_data/Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_2.geojson', filename='Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_2.geojson', extension='.geojson', author='jovyan'),
    ResourceFile(fi

In [16]:
type(graph.get_state(thread).values["gis_scored_files"])

Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add 

list

In [17]:
graph.get_state(thread).values["gis_scored_files"]

Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add 

[GISScoredFile(file_location='data/.DS_Store', filename='.DS_Store', extension='', author='jovyan', gis_relevancy_score=1.0),
 GISScoredFile(file_location='data/README.txt', filename='README.txt', extension='.txt', author='jovyan', gis_relevancy_score=0.0),
 GISScoredFile(file_location='data/layer_data/PFAS sampling and purchasing and intake detail.csv', filename='PFAS sampling and purchasing and intake detail.csv', extension='.csv', author='jovyan', gis_relevancy_score=1.0),
 GISScoredFile(file_location='data/layer_data/WATER_SYSTEMS_IN_KENTUCKY.geojson', filename='WATER_SYSTEMS_IN_KENTUCKY.geojson', extension='.geojson', author='jovyan', gis_relevancy_score=1.0),
 GISScoredFile(file_location='data/layer_data/hot-spot_map.csv', filename='hot-spot_map.csv', extension='.csv', author='jovyan', gis_relevancy_score=1.0),
 GISScoredFile(file_location='data/layer_data/Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_2.geojson', filename='Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_2.geojson', exten

In [18]:
graph.get_state(thread).values["gis_resource_groups"]

Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add 

[GisResourceGroup(resource_name='', entries=[GISScoredFile(file_location='data/.DS_Store', filename='.DS_Store', extension='', author='jovyan', gis_relevancy_score=1.0)], group_score=1.0),
 GisResourceGroup(resource_name='PFAS sampling and purchasing and intake detail', entries=[GISScoredFile(file_location='data/layer_data/PFAS sampling and purchasing and intake detail.csv', filename='PFAS sampling and purchasing and intake detail.csv', extension='.csv', author='jovyan', gis_relevancy_score=1.0), GISScoredFile(file_location='data/layer_data/PFAS sampling and purchasing and intake detail.json', filename='PFAS sampling and purchasing and intake detail.json', extension='.json', author='jovyan', gis_relevancy_score=1.0)], group_score=1.0),
 GisResourceGroup(resource_name='WATER_SYSTEMS_IN_KENTUCKY', entries=[GISScoredFile(file_location='data/layer_data/WATER_SYSTEMS_IN_KENTUCKY.geojson', filename='WATER_SYSTEMS_IN_KENTUCKY.geojson', extension='.geojson', author='jovyan', gis_relevancy_scor

In [19]:
graph.get_state(thread).values.keys()

Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add 

dict_keys(['messages', 'files', 'gis_scored_files', 'gis_resource_groups', 'grouped_resource_entries', 'resource_dataframe_file'])

In [20]:
graph.get_state(thread).values["resource_dataframe_file"]

Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add 

'resource_properties.csv'

In [21]:
import pandas as pd
import duckdb as db

In [22]:
groups_df = pd.read_csv(graph.get_state(thread).values["resource_dataframe_file"])

Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('massi.agents.constructs', 'ResourceFile')]
Deserializing unregistered type massi.agents.constructs.ResourceFile from checkpoint. This will be blocked in a future version. Add 

In [23]:
db.query("""
SELECT * 
FROM groups_df
""").to_df()

,file_location,filename,extension,author,gis_relevancy_score,resource_name,group_score
0,data/.DS_Store,.DS_Store,NaN,jovyan,1.0,NaN,1.0
1,data/layer_data/PFAS sampling and purchasing a...,PFAS sampling and purchasing and intake detail...,.csv,jovyan,1.0,PFAS sampling and purchasing and intake detail,1.0
2,data/layer_data/PFAS sampling and purchasing a...,PFAS sampling and purchasing and intake detail...,.json,jovyan,1.0,PFAS sampling and purchasing and intake detail,1.0
3,data/layer_data/WATER_SYSTEMS_IN_KENTUCKY.geojson,WATER_SYSTEMS_IN_KENTUCKY.geojson,.geojson,jovyan,1.0,WATER_SYSTEMS_IN_KENTUCKY,1.0
4,data/layer_data/WATER_SYSTEMS_IN_KENTUCKY.json,WATER_SYSTEMS_IN_KENTUCKY.json,.json,jovyan,1.0,WATER_SYSTEMS_IN_KENTUCKY,1.0
5,data/layer_data/hot-spot_map.csv,hot-spot_map.csv,.csv,jovyan,1.0,hot-spot_map,1.0
6,data/layer_data/hot-spot_map.json,hot-spot_map.json,.json,jovyan,1.0,hot-spot_map,1.0
7,data/layer_data/Ky_Water_Resources_Polygons_DO...,Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_2.g...,.geojson,jovyan,1.0,Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_2,1.0
8,data/layer_data/Ky_Water_Resources_Polygons_DO...,Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_2.csv,.csv,jovyan,1.0,Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_2,1.0
9,data/layer_data/Ky_Water_Resources_Polygons_DO...,Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_1.g...,.geojson,jovyan,1.0,Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_1,1.0
